# Phase 2: Data Cleaning and Preprocessing

## Objective

The objective of this phase is to improve the quality of the dataset before applying machine learning algorithms. This involves identifying and handling missing values, removing duplicate records, correcting inconsistent data formats, and converting features into a suitable format for analysis and model training.

A clean dataset improves the accuracy, reliability, and performance of machine learning models.

### Step 1: Import Required Libraries

The necessary Python libraries are imported to perform data cleaning and preprocessing operations. Pandas is used for handling the dataset, while NumPy is used for numerical computations.

In [315]:
import pandas as pd 
import numpy as np

### Step 2: Load the Dataset

The dataset is loaded into a Pandas DataFrame to begin the data cleaning process.

In [316]:
df = pd.read_csv("../dataset/Nepali_house_dataset.csv")

### Step 3: Create a Backup Copy

A backup copy of the original dataset is created to ensure that the raw data remains unchanged throughout the preprocessing phase.

In [317]:
df_clean=df.copy()

### Step 4: Identify Missing Values

Before cleaning the dataset, the number of missing values in each feature is examined. This helps determine the appropriate strategy for handling incomplete data.

In [318]:
df_clean.isnull().sum()

TITLE              0
LOCATION           0
PRICE              0
LAND AREA         89
BUILDUP AREA    2699
ROAD ACCESS        9
FACING           206
FLOOR             95
BEDROOM          282
BATHROOM         346
BUILT YEAR        61
PARKING         2786
AMENITIES          0
dtype: int64

In [319]:
missing=(df_clean.isnull().sum()/len(df_clean))*100
missing.sort_values(ascending=True)

TITLE            0.000000
LOCATION         0.000000
PRICE            0.000000
AMENITIES        0.000000
ROAD ACCESS      0.263312
BUILT YEAR       1.784669
LAND AREA        2.603862
FLOOR            2.779403
FACING           6.026916
BEDROOM          8.250439
BATHROOM        10.122879
BUILDUP AREA    78.964307
PARKING         81.509655
dtype: float64

### Step 6: Remove Duplicate Records

Duplicate records can affect model performance and introduce unnecessary bias. Therefore, duplicate entries are identified and removed from the dataset.

In [320]:
print("Before:", df_clean.shape)

df_clean.drop_duplicates(inplace=True)

print("After:", df_clean.shape)

Before: (3418, 13)
After: (3418, 13)


# Cleaning PRICE COLUMN

### Step 1: Analyze the PRICE Column

The **PRICE** column is the target variable of this project. Before training any machine learning model, it is essential to inspect the format of the price values. Since the values may contain different units such as Lakhs, Crores, or Rupees, we first examine the unique formats present in the dataset before converting them into a single numerical representation.

In [321]:
# Display first 20 prices
df_clean["PRICE"].head(20)

0      Rs. 2.9 Cr 
1     Rs. 4.75 Cr 
2     Rs. 1.99 Cr 
3        Rs. 4 Cr 
4     Rs. 12000000
5     Rs. 27000000
6      Rs. 3.3 Cr 
7        Rs. 4 Cr 
8      Rs. 4.5 Cr 
9         Rs. 3 Cr
10    Rs. 4.99 Cr 
11    Rs. 2.45 Cr 
12     Rs. 2.5 Cr 
13    Rs. 4.25 Cr 
14     Rs. 6.5 Cr 
15     Rs. 3.6 Cr 
16     Rs. 5.8 Cr 
17     Rs. 2.5 Cr 
18     Rs. 4.8 Cr 
19    Rs. 6.65 Cr 
Name: PRICE, dtype: str

In [322]:
# Display unique price formats
df_clean["PRICE"].unique()[:30]

<StringArray>
[ 'Rs. 2.9 Cr ', 'Rs. 4.75 Cr ', 'Rs. 1.99 Cr ',    'Rs. 4 Cr ',
 'Rs. 12000000', 'Rs. 27000000',  'Rs. 3.3 Cr ',  'Rs. 4.5 Cr ',
     'Rs. 3 Cr', 'Rs. 4.99 Cr ', 'Rs. 2.45 Cr ',  'Rs. 2.5 Cr ',
 'Rs. 4.25 Cr ',  'Rs. 6.5 Cr ',  'Rs. 3.6 Cr ',  'Rs. 5.8 Cr ',
  'Rs. 4.8 Cr ', 'Rs. 6.65 Cr ', 'Rs. 13.3 Cr ',  'Rs. 8.5 Cr ',
  'Rs. 5.5 Cr ', 'Rs. 2.95 Cr ', 'Rs. 6.35 Cr ', 'Rs. 10.5 Cr ',
  'Rs. 6.4 Cr ',  'Rs. 3.5 Cr ',  'Rs. 2.2 Cr ', 'Rs. 3.75 Cr ',
    'Rs. 8 Cr ', 'Rs. 7.75 Cr ']
Length: 30, dtype: str

In [323]:
# Check data type
df_clean["PRICE"].dtype

<StringDtype(storage='python', na_value=nan)>

In [324]:
# Number of unique price values
df_clean["PRICE"].nunique()

537

In [325]:
df_clean["PRICE"].sample(20, random_state=42)

1964      Rs. 3.45 Cr 
3187      Rs. 2.75 Cr 
170       Rs. 3.45 Cr 
680      Rs. 70,000 /m
2843       Rs. 1.6 Cr 
1763      Rs. 3.85 Cr 
3207       Rs.  3.8 Cr
1765      Rs. 3.35 Cr 
2827       Rs. 3.5 Cr 
2112      Rs. 3.78 Cr 
3237    Rs. 22,500,000
1108       Rs. 3.3 Cr 
194       Rs. 2.55 Cr 
1105      Rs. 2.25 Cr 
70        Rs. 3.25 Cr 
3374          Rs. 5 Cr
2321       Rs. 3.6 Cr 
1336        Rs. 21 Cr 
969       Rs. 1.55 Cr 
1621      Rs. 3.38 Cr 
Name: PRICE, dtype: str

### Step 2: Identify Rental Listings

The dataset is intended for predicting **house sale prices**. However, some records represent **monthly rental prices**, identified by the presence of `/m` in the `PRICE` column. These records are identified before being removed from the dataset to ensure consistency in the target variable.

In [326]:
# Find rows containing "/m"
rent_houses = df_clean[df_clean["PRICE"].str.contains("/m", na=False)]

rent_houses

,TITLE,LOCATION,PRICE,LAND AREA,BUILDUP AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
98,4 BHK House for Rent,"Bhaisepati, Lalitpur","Rs. 65,000 /m",4.0 aana,NaN,20 Feet,North-West,2.5,4.0,3.0,2070 B.S,1 CaRs. & 2 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
109,5 BHK Bungalow for Rent,"Budhanilkantha, Kathmandu",Rs. 1.4 Lac/m,12.1 aana,NaN,16 Feet,East,2.5,5.0,5.0,2074 B.S,3 CaRs. & 3 Bikes,"['Marbel', 'Parquet', 'Earthquake Resistant', ..."
111,House for Rent,"Khumaltar, Lalitpur",Rs. 1.5 Lac/m,16.0 aana,NaN,20 Feet,NaN,2.0,6.0,3.0,2076 B.S,6 CaRs.,"['Marbel', 'Drainage', 'Garden', 'Parking', 'T..."
127,House for Rent,"Ranibari, Kathmandu",Rs. 1.05 Lac/m,5.1 aana,3119.28 Sq. Ft.,12 Feet,South-West,3.5,7.0,5.0,2074 B.S,2 CaRs.,"['Marbel', 'Drainage', 'Parking', 'Drinking Wa..."
159,House for Rent,"Naxal, Kathmandu",Rs. 2.5 Lac/m,24.0 aana,1096 Sq. Ft.,15 Feet,South,2.5,6.0,4.0,2073 B.S,10 CaRs. & 3 Bikes,"['Earthquake Resistant', 'Marbel', 'Parquet', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3151,House for rent at Bhaisepati,"Bhaisepati, Lalitpur",Rs. 3 Lac/m,16.0 aana,NaN,0 Feet,South East,2.5,8.0,7.0,2071 B.S,NaN,"['Wifi', 'Drainage', 'Water Supply', 'Water Ta..."
3152,"House for rent at Sanepa, Lalitpur","Sanepa, Lalitpur",Rs. 1.75 Lac/m,8.0 aana,NaN,0 Feet,South,2.5,4.0,5.0,2076 B.S,NaN,['Parking']
3167,House for rent at Maharajgunj,"Maharajgunj, Kathmandu",Rs. 4.5 Lac/m,NaN,NaN,200 Feet,NaN,2.5,1.0,1.0,2070 B.S,NaN,"['Parquet', 'Marbel', 'Bathroom', 'Parking', '..."
3171,House on rent at Chappalkarkhana,"Chappal karkhana, Kathmandu",Rs. 3.3 Lac/m,24.0 aana,NaN,18 Feet,NaN,2.5,1.0,1.0,NaN,NaN,"['Bathroom', 'Drainage', 'Parking']"


In [327]:
# Find prices containing "Lac"
df_clean[df_clean["PRICE"].str.contains("Lac", case=False, na=False)]["PRICE"].unique()

<StringArray>
['Rs. 50 Lac/aana',   'Rs. 1.4 Lac/m',   'Rs. 1.5 Lac/m',  'Rs. 1.05 Lac/m',
 'Rs. 90 Lac/aana',   'Rs. 2.5 Lac/m', 'Rs. 70 Lac/aana', 'Rs. 21 Lac/aana',
   'Rs. 3.5 Lac/m',   'Rs. 1.6 Lac/m',
 ...
   'Rs. 2.68 Lac ',   'Rs. 98.5 Lac ',  'Rs. 1.99 Lac/m',  'Rs. 2.35 Lac/m',
 'Rs. 68 Lac/aana',     'Rs. 82 Lac ',   'Rs. 1.99 Lac ',   'Rs. 4.2 Lac/m',
 'Rs. 64 Lac/aana',    'Rs. 2.5 Lac ']
Length: 111, dtype: str

In [328]:
# Find prices containing "/"
df_clean[df_clean["PRICE"].str.contains("/", na=False)]["PRICE"].unique()

<StringArray>
['Rs. 50 Lac/aana',   'Rs. 65,000 /m',   'Rs. 1.4 Lac/m',   'Rs. 1.5 Lac/m',
  'Rs. 1.05 Lac/m', 'Rs. 90 Lac/aana',   'Rs. 2.5 Lac/m', 'Rs. 70 Lac/aana',
 'Rs. 21 Lac/aana',   'Rs. 60,000 /m',
 ...
  'Rs. 9.89 Lac/m',  'Rs. 1.49 Lac/m',   'Rs. 2.8 Lac/m',  'Rs. 2.55 Lac/m',
 'Rs. 62 Lac/aana',  'Rs. 1.99 Lac/m',  'Rs. 2.35 Lac/m', 'Rs. 68 Lac/aana',
   'Rs. 4.2 Lac/m', 'Rs. 64 Lac/aana']
Length: 108, dtype: str

In [329]:
# Find all non-Crore prices
df_clean[~df_clean["PRICE"].str.contains("Cr", na=False)]["PRICE"].unique()[:50]

<StringArray>
[   'Rs. 12000000',    'Rs. 27000000', 'Rs. 50 Lac/aana',   'Rs. 65,000 /m',
   'Price on call',   'Rs. 1.4 Lac/m',   'Rs. 1.5 Lac/m',  'Rs. 1.05 Lac/m',
 'Rs. 90 Lac/aana', 'Rs. 3,88,000,00',   'Rs. 2.5 Lac/m', 'Rs. 70 Lac/aana',
 'Rs. 21 Lac/aana',   'Rs. 60,000 /m',   'Rs. 3.5 Lac/m',   'Rs. 80,000 /m',
   'Rs. 70,000 /m',   'Rs. 45,000 /m',  'Rs. 27500000  ',   'Rs. 1.6 Lac/m',
  'Rs. 1.28 Lac/m',    'Rs. 1.5 Lac ',   'Rs. 90,000 /m',   'Rs. 1.7 Lac/m',
  'Rs. 1.85 Lac/m',     'Rs. 3 Lac/m',  'Rs. 1.55 Lac/m',     'Rs. 2 Lac/m',
   'Rs. 82,000 /m',   'Rs. 1.3 Lac/m',     'Rs. 1 Lac/m',   'Rs. 2.2 Lac/m',
     'Rs. 5 Lac/m',  'Rs. 1.75 Lac/m',   'Rs. 75,000 /m',     'Rs. 90 Lac ',
   'Rs. 1.1 Lac/m',     'Rs. 4 Lac/m',     'Rs. 6 Lac/m',  'Rs. 1.65 Lac/m',
     'Rs. 65 Lac ',  'Rs. 2.25 Lac/m',  'Rs. 3.35 Lac/m',   'Rs. 4.5 Lac/m',
  'Rs. 1.25 Lac/m',  'Rs. 12.7 Lac/m',  'Rs. 2.75 Lac/m',   'Rs. 85,000 /m',
     'Rs. 15 Lac ',  'Price on call ']
Length: 50, dtype: str

### Step 3: Remove Invalid Price Records

The dataset contains different types of price representations, including monthly rent, price per Aana, price per square foot, and listings with unavailable prices ("Price on call"). Since the objective of this project is to predict the total selling price of houses, these records are removed to maintain consistency in the target variable.

In [330]:
invalid_patterns = [
    "/m",
    "/aana",
    "/sf",
    "Price on call"
]

mask = ~df_clean["PRICE"].str.contains(
    "|".join(invalid_patterns),
    case=False,
    na=False
)

df_clean = df_clean[mask]
df_clean.shape

(2667, 13)

### Step 4: Convert Price into Numerical Format

The values in the **PRICE** column are stored in different textual formats such as Crores and Rupees with commas. Machine learning algorithms require numerical values, so all prices are converted into a single numerical representation in Nepalese Rupees.

In [331]:
import re
def clean_price(price):
    price = str(price).strip()

    #Remove "Rs."
    price=price.replace("Rs.","").strip()

    #Remove commas
    price=price.replace(",","")

    #convert crores to rupess
    if "Cr" in price:
        price = price.replace("Cr","").strip()
        return float(price)*10000000

     # Lakhs
    elif "Lac" in price:
        value = float(price.replace("Lac", "").strip())
        return value * 100000

    # Already numeric
    else:
        return float(price)

df_clean["PRICE"]=df_clean["PRICE"].apply(clean_price)

In [332]:
df_clean["PRICE"].head(50)

0      29000000.0
1      47500000.0
2      19900000.0
3      40000000.0
4      12000000.0
5      27000000.0
6      33000000.0
7      40000000.0
8      45000000.0
9      30000000.0
10     49900000.0
11     24500000.0
12     25000000.0
13     42500000.0
14     65000000.0
15     36000000.0
16     58000000.0
17     25000000.0
18     48000000.0
19     66500000.0
20    133000000.0
21     85000000.0
22     55000000.0
23     47500000.0
24     29500000.0
25     63500000.0
26     85000000.0
27    105000000.0
28     64000000.0
29     35000000.0
30     22000000.0
31     37500000.0
32     80000000.0
33     55000000.0
34     55000000.0
35     77500000.0
36    160000000.0
37     31900000.0
38     25000000.0
39    120000000.0
40     62500000.0
41    160000000.0
42     28000000.0
43     59900000.0
44     48000000.0
45     25000000.0
46     33500000.0
47     35000000.0
48     40000000.0
49     60000000.0
Name: PRICE, dtype: float64

# Cleaning LAND AREA COLUMN

### Step 1: Analyze the LAND AREA Column

The **LAND AREA** column represents the total land size of the property. The values are stored using different measurement units such as Aana, Kattha, Dhur, Ropani, Square Feet, and Square Meter. Before applying machine learning algorithms, all land area values must be converted into a single standard unit.

In [333]:
df_clean["LAND AREA"].sample(50, random_state=42)

354        3.2 aana
2384       5.3 aana
2111       4.0 aana
3198       4.0 aana
1223       3.3 aana
673        3.0 aana
775        3.2 aana
659        3.0 aana
2057       4.2 aana
1587       4.0 aana
3034       5.0 aana
2786       4.0 aana
3311         3 aana
2828       5.1 aana
343     0.12 kattha
1681       6.0 aana
830        3.3 aana
2831       4.0 aana
2938       3.2 aana
2376       6.0 aana
1769       3.0 aana
3226       4.1 aana
2203       8.0 aana
584        4.1 aana
826        3.1 aana
825        4.2 aana
1602       3.2 aana
1332       5.0 aana
3264       4.2 aana
1484         3 aana
1937       4.3 aana
536        2.2 aana
3286        4 aana 
32         6.0 aana
774        3.0 aana
3196      12.0 aana
1489       8.0 aana
2671       6.0 aana
2455       4.0 aana
3387       4.7 aana
627        2.3 aana
2210      10.0 aana
73         2.3 aana
521        2.5 aana
70         4.3 aana
1447       3.0 aana
725        2.3 aana
1703      10.0 aana
2025       9.2 aana
1416       3.2 aana


In [334]:
df_clean["LAND AREA"].dropna().unique()[:100]

<StringArray>
[    '4.0 aana',     '3.0 aana',     '2.3 aana',     '7.0 aana',
     '6.0 aana',     '3.2 aana',     '4.3 aana',     '2.2 aana',
     '5.0 aana',     '5.2 aana',     '6.2 aana',     '4.2 aana',
     '4.1 aana',     '9.6 aana',     '3.1 aana',    '11.0 aana',
     '3.3 aana',     '9.0 aana',      '12 aana',     '6.4 aana',
    '12.0 aana',     '3.5 aana',       '6 aana',     '7.1 aana',
     '6.3 aana',     '9.3 aana',     '9.1 aana',     '6.1 aana',
     '2.9 aana',     '7.2 aana',    '10.2 aana',     '8.0 aana',
    '15.0 aana',     '7.3 aana',     '5.1 aana',    '10.0 aana',
     '5.3 aana',     '2.5 aana',  '0.12 kattha',   '0.5 kattha',
    '14.0 aana',     '1.2 aana',     '8.2 aana',     '1.1 aana',
     '3.8 aana',    '13.0 aana',    '16.0 aana', '1.9.9 kattha',
  '0.11 kattha',     '9.2 aana',     '3.4 aana',     '2.8 aana',
    '17.0 aana',   '1.7 kattha',   '1.3 kattha',       '3 aana',
    '11.2 aana',    '10.3 aana',  '0.15 kattha',  '0.14 kattha',
   '3.2 kat

In [335]:
# Count different units
print(df_clean["LAND AREA"].str.contains("aana", case=False, na=False).sum())

print(df_clean["LAND AREA"].str.contains("kattha", case=False, na=False).sum())

print(df_clean["LAND AREA"].str.contains("sq", case=False, na=False).sum())

2609
27
11


In [336]:
df_clean[
    df_clean["LAND AREA"].str.contains(r"\.\d+\.\d+", regex=True, na=False)
]["LAND AREA"]

463    1.9.9 kattha
Name: LAND AREA, dtype: str

### Step 7: Standardize the LAND AREA Feature

The **LAND AREA** feature contains measurements in multiple units such as **Aana**, **Kattha**, and **Square Feet**. Machine learning models require numerical values in a consistent unit. Therefore, all land area measurements are converted into **Square Feet (sq.ft)** using standard Nepali land conversion factors.

Conversion factors used:

- 1 Aana = 342.25 sq.ft
- 1 Kattha = 3645 sq.ft
- 1 Square Foot = 1 sq.ft

In [337]:
import re

def convert_land_area(area):
    if pd.isna(area):
        return np.nan

    area = str(area).strip().lower()

    # Remove extra spaces
    area = re.sub(r"\s+", " ", area)

    # Fix known typo
    area = area.replace("1.9.9", "1.99")

    # Invalid value
    if area == "0 sq. ft":
        return np.nan

    # Aana → Square Feet
    if "aana" in area:
        value = float(area.replace("aana", "").strip())
        return value * 342.25

    # Kattha → Square Feet
    elif "kattha" in area:
        value = float(area.replace("kattha", "").strip())
        return value * 3645

    # Square Feet
    elif "sq. ft" in area:
        value = float(area.replace("sq. ft", "").strip())
        return value

    return np.nan
df_clean["LAND AREA"] = df_clean["LAND AREA"].apply(convert_land_area)

In [338]:
df_clean[["LAND AREA"]].head(20)

,LAND AREA
0,1369.000
1,1026.750
2,787.175
3,2395.750
4,2053.500
5,2053.500
6,1095.200
7,1471.675
8,752.950
9,1711.250


In [339]:
df_clean["LAND AREA"].dtype

dtype('float64')

In [340]:
df_clean["LAND AREA"].isnull().sum()

np.int64(21)

In [341]:
df_clean["LAND AREA"].describe()

count     2646.000000
mean      1730.681679
std       1403.189769
min          4.500000
25%       1095.200000
50%       1369.000000
75%       1779.700000
max      36450.000000
Name: LAND AREA, dtype: float64

In [342]:
df_clean[df_clean["LAND AREA"] < 100]

,TITLE,LOCATION,PRICE,LAND AREA,BUILDUP AREA,ROAD ACCESS,FACING,FLOOR,BEDROOM,BATHROOM,BUILT YEAR,PARKING,AMENITIES
1783,"Commercial building for sale in Machhapokhari,...","Machhapokhari, Kathmandu",55000000.0,4.5,9500 Sq. Feet,20 Feet,South-East,5.5,12.0,6.0,2076 B.S,NaN,"['Drainage', 'Drinking Water', 'Power Backup',..."


In [343]:
# Remove unrealistic land area values
df_clean = df_clean[df_clean["LAND AREA"] >= 100]

In [344]:
df_clean["LAND AREA"].describe()

count     2645.00000
mean      1731.33430
std       1403.05339
min        102.67500
25%       1095.20000
50%       1369.00000
75%       1779.70000
max      36450.00000
Name: LAND AREA, dtype: float64

# Inspect Remaining Columns

In [345]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 2645 entries, 0 to 3417
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TITLE         2645 non-null   str    
 1   LOCATION      2645 non-null   str    
 2   PRICE         2645 non-null   float64
 3   LAND AREA     2645 non-null   float64
 4   BUILDUP AREA  562 non-null    str    
 5   ROAD ACCESS   2643 non-null   str    
 6   FACING        2586 non-null   str    
 7   FLOOR         2606 non-null   float64
 8   BEDROOM       2469 non-null   float64
 9   BATHROOM      2413 non-null   float64
 10  BUILT YEAR    2622 non-null   str    
 11  PARKING       542 non-null    str    
 12  AMENITIES     2645 non-null   str    
dtypes: float64(5), str(8)
memory usage: 289.3 KB


In [346]:
missing = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing %": round(df_clean.isnull().sum() / len(df_clean) * 100, 2)
})

missing.sort_values("Missing %", ascending=False)

,Missing Count,Missing %
PARKING,2103,79.51
BUILDUP AREA,2083,78.75
BATHROOM,232,8.77
BEDROOM,176,6.65
FACING,59,2.23
FLOOR,39,1.47
BUILT YEAR,23,0.87
ROAD ACCESS,2,0.08
TITLE,0,0.00
LAND AREA,0,0.00


# Clean BUILDUP Area

The **BUILDUP AREA** column contained approximately 80% missing values. Since such a large proportion of missing data would make imputation unreliable and could introduce bias into the model, the feature was removed from the dataset.

In [347]:
df_clean.drop(columns=["BUILDUP AREA"], inplace=True)

# Clean ROAD ACCESS

In [348]:
df_clean["ROAD ACCESS"].dropna().unique()[:50]

<StringArray>
[   '12 Feet',    '10 Feet',    '20 Feet',    '13 Feet',    '14 Feet',
    '16 Feet',    '26 Feet',    '25 Feet',    '19 Feet',    '18 Feet',
    '15 Feet',    '24 Feet',    '27 Feet',    '30 Feet',    '23 Feet',
     '8 Feet',    '11 Feet',    '22 Feet',   '20 Meter',     '6 Feet',
     '4 Feet',   '10 Meter',    '17 Feet',   '13 Meter', '13-20 Feet',
 '12-18 Feet', '10-12 Feet', '10-15 Feet', '12-16 Feet',    '32 Feet',
 '12/20 Feet',     '9 Feet', '12-14 Feet', '15-26 Feet', '10-20 Feet',
 '14-20 Feet', '12-13 Feet',  '8-10 Feet', '12/13 Feet',   '20  Feet',
  '9-12 Feet',   '13  Feet', '10/13 Feet', '10-24 Feet', '16-22 Feet',
 '20-26 Feet', '13-16 Feet', '12-15 Feet',  '8-12 Feet', '15-24 Feet']
Length: 50, dtype: str

In [351]:
import re

def clean_road(value):

    # Missing value
    if pd.isna(value):
        return np.nan

    # Convert everything to string
    value = str(value).lower().strip()

    # Replace "/" with "-"
    value = value.replace("/", "-")

    # Extract all numbers
    numbers = re.findall(r"\d+\.?\d*", value)

    if len(numbers) == 0:
        return np.nan

    numbers = [float(x) for x in numbers]

    # Average if there are multiple numbers
    road = sum(numbers) / len(numbers)

    # Convert meters to feet
    if "meter" in value:
        road *= 3.28084

    return road

df_clean["ROAD ACCESS"] = df_clean["ROAD ACCESS"].apply(clean_road)

In [352]:
df_clean["ROAD ACCESS"].head(20)

0     12.0
1     10.0
2     10.0
3     12.0
4     20.0
5     12.0
6     13.0
7     14.0
8     10.0
9     12.0
10    13.0
11    16.0
12    13.0
13    20.0
14    12.0
15    13.0
16    14.0
17    12.0
18    16.0
19    20.0
Name: ROAD ACCESS, dtype: float64

In [353]:
df_clean["ROAD ACCESS"].describe()

count    2643.000000
mean       14.689400
std         4.833321
min         0.000000
25%        12.000000
50%        13.000000
75%        16.000000
max        82.000000
Name: ROAD ACCESS, dtype: float64